In [1]:
from CGCNN_MT.datamodule.prepare_data import main as prepare_data
from CGCNN_MT.datamodule.clean_cif import main as clean_cif
from pathlib import Path
import os
import shutil
from CGCNN_MT.slurm_sub import run_slurm_job
import pandas as pd
from CGCNN_MT.datamodule.dataset import LoadGraphData
import json

The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.


In [2]:
src_cif_dir = Path("raw_data/CoREMOF2019").absolute()
tgt_cif_dir = Path("CGCNN_MT/data/CoREMOF2019/clean_cifs").absolute()
tgt_cif_dir.mkdir(exist_ok=True, parents=True)

In [3]:
slrum_template = """#!/bin/bash
#SBATCH --job-name={job_name}
#SBATCH --output={work_dir}/%x_%A.out
#SBATCH --error={work_dir}/%x_%A.err
#SBATCH --partition=C9654 
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task={n_cpus}
#SBATCH --mem-per-cpu=2G

export PATH=/opt/share/miniconda3/envs/mofmthnn/bin/:$PATH
export LD_LIBRARY_PATH=/opt/share/miniconda3/envs/mofmthnn/lib/:$LD_LIBRARY_PATH

srun python -u {python_script} --cif_dir {src_cif_dir} --output_dir {output_dir} --sanitize True --log_file {log_file} --n_cpus {n_cpus}
"""
python_script = Path("CGCNN_MT/datamodule/clean_cif.py").absolute()
job_name = "clean_cif_CoREMOF2019"
work_dir = tgt_cif_dir.parent
n_cpus = 1
src_cif_dir = str(src_cif_dir)
output_dir = str(tgt_cif_dir)
log_file = str(tgt_cif_dir.parent / "clean_cif.log")
slrum_script = slrum_template.format(job_name=job_name, work_dir=work_dir, n_cpus=n_cpus, src_cif_dir=src_cif_dir, output_dir=output_dir, log_file=log_file, python_script=python_script)

with open(tgt_cif_dir.parent / "clean_cif.sh", "w") as f:
    f.write(slrum_script)

In [4]:
process = run_slurm_job(work_dir, executor="sbatch", script_name="clean_cif.sh")
## get the output of the job
while True:
    output = process.stdout.readline()
    err = process.stderr.readline()
    if err:
        print(err.decode().strip())
        break
    if output == b'' and process.poll() is not None:
        break
    if output:
        print(output.decode().strip())
print(f"Submitted job {job_name} with PID {process.pid}")

Submitted batch job 199759
Submitted job clean_cif_CoREMOF2019 with PID 535509


In [21]:
len(os.listdir(src_cif_dir))

24040

In [ ]:
src_cif_dir = Path("raw_data/CoREMOF2019").absolute()
tgt_cif_dir = Path("CGCNN_MT/data/CoREMOF2019/clean_cifs").absolute()
tgt_cif_dir.mkdir(exist_ok=True, parents=True)

slrum_template = """#!/bin/bash
#SBATCH --job-name={job_name}
#SBATCH --output={work_dir}/%x_%A.out
#SBATCH --error={work_dir}/%x_%A.err
#SBATCH --partition=C9654 
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task={n_cpus}
#SBATCH --mem-per-gpu=100G
#SBATCH --gres=gpu:1

export PATH=/opt/share/miniconda3/envs/mofmthnn/bin/:$PATH
export LD_LIBRARY_PATH=/opt/share/miniconda3/envs/mofmthnn/lib/:$LD_LIBRARY_PATH

srun python -u {python_script} --cif_dir {src_cif_dir} --n_cpus {n_cpus}
"""
python_script = Path("CGCNN_MT/inference.py").absolute()
job_name = "inference_CoREMOF2019"
work_dir = tgt_cif_dir.parent
n_cpus = 64
src_cif_dir = str(src_cif_dir)
output_dir = str(tgt_cif_dir)
log_file = str(tgt_cif_dir.parent / "inference.log")
slrum_script = slrum_template.format(job_name=job_name, work_dir=work_dir, n_cpus=n_cpus, src_cif_dir=src_cif_dir, output_dir=output_dir, log_file=log_file, python_script=python_script)

with open(tgt_cif_dir.parent / "inference.sh", "w") as f:
    f.write(slrum_script)

In [3]:
notes = "CoREMOF2019"
pred_file = Path(f"CGCNN_MT/inference/{notes}/infer_results_version_43.csv")
pred_df = pd.read_csv(pred_file)
pred_df.head()

,MofName,TSD_pred,TSD_uncertainty,SSD_pred,SSD_uncertainty,WS24_water_pred,WS24_water_uncertainty,WS24_water4_pred,WS24_water4_uncertainty,WS24_acid_pred,WS24_acid_uncertainty,WS24_base_pred,WS24_base_uncertainty,WS24_boiling_pred,WS24_boiling_uncertainty
0,1499489-acs.cgd.6b01265_1499490_clean,322.7797,0.1197,1,0.7359,1,0.8345,2,0.9845,0,0.2160,0,0.0967,0,0.1608
1,ABAVIJ_clean,354.5818,0.2831,1,-0.0334,0,0.9944,1,0.9059,0,0.2954,0,0.2274,0,0.1746
2,ABAYIO_clean,427.0172,0.5067,1,-0.0334,1,0.8176,0,1.0000,1,0.8403,1,0.8725,1,0.5904
3,ABAYOU_clean,373.6390,0.0956,1,0.1574,0,0.6107,0,0.9996,0,0.9187,0,0.5462,0,0.1458
4,ABEFUL_clean,336.7863,0.1629,1,0.5586,1,0.1585,2,0.6228,1,0.0258,1,0.0181,1,0.0473


In [4]:
pred_df["WS24_water4_pred"].unique()

array([2, 1, 0, 3])

In [5]:
data_root_dir = "./CGCNN_MT/data"
data_root_dir = Path(data_root_dir)
ssd_raw_csv = "./raw_data/Nandy_2022_SciData/separate_files/solvent_removal_stability/full_SSD_data.csv"
tsd_raw_csv = "./raw_data/Nandy_2022_SciData/separate_files/thermal_stability/full_TSD_data.csv"

mof_name_map = {}

df_raw_ssd = pd.read_csv(ssd_raw_csv)
df_raw_tsd = pd.read_csv(tsd_raw_csv)
df_raw_ssd.dropna(inplace=True)
df_raw_tsd.dropna(inplace=True)
for df in [df_raw_ssd, df_raw_tsd]:
    for i, row in df.iterrows():
        mof_name = row["CoRE_name"]
        refcode = row["refcode"]
        if mof_name not in mof_name_map:
            mof_name_map[mof_name] = refcode
        else:
            assert mof_name_map[mof_name] == refcode, f"{mof_name} has multiple refcodes: {refcode} and {mof_name_map[mof_name]}"
print(f"total {len(df_raw_ssd) + len(df_raw_tsd)} samples")
print(f"total {len(mof_name_map)} mofs")

tasks = ["TSD", "SSD", "WS24_water", "WS24_water4", "WS24_acid", "WS24_base", "WS24_boiling"]
class_map_2 = {0: "unstable", 1: "stable"}
class_map_4 = {0: "U", 1: "LK", 2: "HK", 3: "TS"}

dfs = {}
for task in tasks:
    data_dir = data_root_dir / task
    split_dfs = {}
    for split in ["train", "val", "test"]:
        dataset = LoadGraphData(data_dir, split, csv_file_name="id_prop_feat.csv")
        split_dfs[split] = dataset.id_prop_df[["Partition"] + dataset.prop_cols].copy()
        split_dfs[split].rename(columns={dataset.prop_cols[0]: "Label"}, inplace=True)
        if task in ["TSD", "SSD"]:
            split_dfs[split].index = pd.Series(split_dfs[split].index).apply(lambda x: mof_name_map[x])
        if len(split_dfs[split]["Label"].unique()) == 2:
            split_dfs[split]["Label"] = split_dfs[split]["Label"].apply(lambda x: class_map_2[int(x)])
        elif len(split_dfs[split]["Label"].unique()) == 4:
            split_dfs[split]["Label"] = split_dfs[split]["Label"].apply(lambda x: class_map_4[int(x)])
    dfs.update({task: split_dfs})
    dfs[task]["total"] = pd.concat(split_dfs.values())
all_training_mofs = []
for task in tasks:
    df = dfs[task]["total"]
    all_training_mofs.extend(df.index.tolist())
all_training_mofs = list(set(all_training_mofs))
print(len(all_training_mofs))
print(all_training_mofs[:10])

total 5311 samples
total 4148 mofs
prop_cols: ['Label']
prop_cols: ['Label']
prop_cols: ['Label']
prop_cols: ['Label']
prop_cols: ['Label']
prop_cols: ['Label']
prop_cols: ['water_label']
prop_cols: ['water_label']
prop_cols: ['water_label']
prop_cols: ['water4_label']
prop_cols: ['water4_label']
prop_cols: ['water4_label']
prop_cols: ['acid_label']
prop_cols: ['acid_label']
prop_cols: ['acid_label']
prop_cols: ['base_label']
prop_cols: ['base_label']
prop_cols: ['base_label']
prop_cols: ['boiling_label']
prop_cols: ['boiling_label']
prop_cols: ['boiling_label']
4963
['QOYLOG', 'TOHYOF', 'ROCZUG', 'GEDLIM', 'CELZIE', 'EPECEJ', 'HOFJIX', 'FIFPOA', 'EDOMAM', 'WIFGOJ']


In [11]:
uncertainty_cutoff = {
    "TSD_uncertainty": 0.5,
    # "SSD_uncertainty": 0.5,
    # "WS24_water_uncertainty": 0.5,
    # "WS24_water4_uncertainty": 0.5
}
thermal_cutoff = 300

hit_df = pred_df.copy()
# hit_df = pred_df[(pred_df["SSD_pred"]==1)&(pred_df["WS24_water_pred"]==1)&(pred_df["WS24_water4_pred"]>1)]
# print("Hits after solvent stability and water stability filetering:", len(hit_df))

hit_df = hit_df[hit_df["TSD_pred"]>thermal_cutoff]
print(f"Hits after thermal stability of {thermal_cutoff} ℃ filetering: ", len(hit_df))

# hit_df3 = hit_df2
for key in uncertainty_cutoff:
    hit_df = hit_df[(hit_df[key]<uncertainty_cutoff[key])]
print("Hits after uncertainty cutoffs filetering:", len(hit_df))
hit_df = hit_df.copy()
hit_df.insert(1, "RefCode", hit_df["MofName"].apply(lambda x: x.split("_")[0]))
hit_df = hit_df[~hit_df["RefCode"].isin(all_training_mofs)]
print("Hits after filetering training samples:", len(hit_df))
hit_df

Hits after thermal stability of 300 ℃ filetering:  9703
Hits after uncertainty cutoffs filetering: 9136
Hits after filetering training samples: 5831


,MofName,RefCode,TSD_pred,TSD_uncertainty,SSD_pred,SSD_uncertainty,WS24_water_pred,WS24_water_uncertainty,WS24_water4_pred,WS24_water4_uncertainty,WS24_acid_pred,WS24_acid_uncertainty,WS24_base_pred,WS24_base_uncertainty,WS24_boiling_pred,WS24_boiling_uncertainty
0,1499489-acs.cgd.6b01265_1499490_clean,1499489-acs.cgd.6b01265,322.7797,0.1197,1,0.7359,1,0.8345,2,0.9845,0,0.2160,0,0.0967,0,0.1608
7,ABETIN_clean,ABETIN,317.2982,0.1202,1,0.3530,1,0.0364,2,0.0558,0,0.0730,0,0.0829,1,0.8868
17,ABIYIV_clean,ABIYIV,329.1877,0.0595,1,0.1847,0,0.0433,1,0.2237,0,0.1654,0,0.0912,0,0.0696
18,ABULOB_clean,ABULOB,318.1038,0.0481,0,0.0528,1,0.1710,2,0.1018,0,0.2542,0,0.0544,0,0.2478
19,ABUWOJ_clean,ABUWOJ,414.6467,0.1237,1,0.7308,0,0.1111,0,0.1638,0,0.1572,0,0.7179,0,0.1251
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12008,magnetochemistry3010001_PF362NdCl_clean,magnetochemistry3010001,319.0383,0.4376,1,0.9973,1,0.2117,3,0.5755,0,0.4591,0,0.2148,1,0.1484
12009,nchem.2258-s3_clean,nchem.2258-s3,407.4680,0.1792,0,0.1653,1,0.8013,2,0.7751,0,0.4127,1,0.8916,1,0.9667
12010,ncomms11831_ncomms11831-s2_clean,ncomms11831,527.1912,0.4475,0,0.9971,0,0.1472,1,0.7959,0,0.8611,0,0.8254,0,0.9779
12011,ncomms11831_ncomms11831-s3_clean,ncomms11831,524.1849,0.4736,0,0.9520,0,0.9862,1,0.9979,0,0.9644,0,0.6919,0,0.9349


In [12]:
exclude_df = pd.read_csv("/home/zhangsd/repos/MOFSNN/CGCNN_MT/inference/CoREMOF2019/stable_coremof_1225.csv")
exclude_df.head(2)

,MofName,RefCode,TSD_pred,TSD_uncertainty,SSD_pred,SSD_uncertainty,WS24_water_pred,WS24_water_uncertainty,WS24_water4_pred,WS24_water4_uncertainty,WS24_acid_pred,WS24_acid_uncertainty,WS24_base_pred,WS24_base_uncertainty,WS24_boiling_pred,WS24_boiling_uncertainty
0,ABETIN_clean,ABETIN,317.2982,0.1202,1,0.3530,1,0.0364,2,0.0558,0,0.073,0,0.0829,1,0.8868
1,ACECIV_ion_b,ACECIV,334.9983,0.1665,1,0.0541,1,0.0728,2,0.1211,0,0.087,0,0.1066,0,0.0002


In [13]:
hit_df = hit_df[~hit_df["MofName"].isin(exclude_df["MofName"])]
print(hit_df.shape)

(4606, 16)


In [16]:
hit_df.to_csv(pred_file.parent / f'stable_coremof_{len(hit_df)}.csv', index=False)
hit_cif_dir = (pred_file.parent / f'stable_coremof_{len(hit_df)}').absolute()
src_cif_dir = src_cif_dir.absolute()
hit_cif_dir.mkdir(exist_ok=True)
for name in hit_df['MofName']:
    # shutil.copy(src_cif_dir / f'{name}.cif', hit_cif_dir)
    os.symlink(src_cif_dir / f'{name}.cif', hit_cif_dir / f'{name}.cif', target_is_directory=False)